In [ ]:
from sklift.datasets import fetch_lenta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from catboost import CatBoostClassifier, Pool
import shap

from causalml.metrics import plot_gain, auuc_score
from causalml.inference.tree import UpliftTreeClassifier, UpliftRandomForestClassifier
from causalml.inference.meta import BaseDRClassifier, BaseRClassifier, BaseSClassifier, BaseTClassifier, BaseXClassifier
from causalml.inference.iv import BaseDRIVRegressor
import optuna

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import auc
from IPython.display import clear_output

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
print(sys.executable)

In [ ]:
SEED = 42

In [ ]:
BASE_MODEL = CatBoostClassifier(
    loss_function='Logloss',
    verbose=False,
    random_state=SEED
  )

In [ ]:
dataset = fetch_lenta()
data, target, treatment = dataset.data, dataset.target, dataset.treatment

In [ ]:
# Преобразование признака gender
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
data['gender'] = data['gender'].fillna(-1)

In [ ]:
# Заполнение пропусков
data = data.fillna(-100)

In [ ]:
# Удаление явных дубликатов
data = data.drop_duplicates()

In [ ]:
# Разбиение на train, valid, test выборки

data['treatment'] = treatment
data['treatment'] = (data['treatment'] == 'test').astype(int)
data['target'] = target

train_val_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=SEED, stratify=data[['treatment', 'target']])

train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=SEED, stratify=data.loc[train_val_idx, ['treatment', 'target']])

X_train = data.drop(columns=['treatment', 'target']).loc[train_idx]
X_val = data.drop(columns=['treatment', 'target']).loc[val_idx]
X_test = data.drop(columns=['treatment', 'target']).loc[test_idx]

treatment_train = data['treatment'][train_idx]
treatment_val = data['treatment'][val_idx]
treatment_test = data['treatment'][test_idx]

y_train = target[train_idx]
y_val = target[val_idx]
y_test = target[test_idx]

In [ ]:
def get_uplift(model_t, model_c, X, treatment, y, selected_features):

  X_selected = X[selected_features]

  pred_c = model_c.predict_proba(X_selected)[:, 1]
  pred_t = model_t.predict_proba(X_selected)[:, 1]

  uplift_pred = pred_t - pred_c

  pred_data = pd.DataFrame({'uplift': uplift_pred,
                            'y': y,
                            'treatment': treatment})

  auuc = auuc_score(pred_data, outcome_col='y', treatment_col='treatment')
  # auuc = plot_auuc_curve(y, treatment, uplift_pred, draw_auuc=draw_auuc)

  print('AUUC:', auuc.squeeze())

  return uplift_pred, auuc

In [ ]:
def train_tlearner(base_model, X_train, treatment_train, y_train,
                   X_val, treatment_val, y_val,
                   selected_features, n_trials=50):

    X_train = pd.concat([X_train, treatment_train], axis=1)
    y_train = pd.concat([y_train, treatment_train], axis=1)

    base_params = {
        'loss_function': 'Logloss',
        'verbose': False,
        'random_state': SEED
    }

    def objective(trial):

        params_c = {
            'depth': trial.suggest_int('depth_c', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate_c', 0.01, 0.3, log=True),
            'iterations': trial.suggest_int('iterations_c', 300, 1500),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_c', 1, 20),
            'bagging_temperature': trial.suggest_float('bagging_temperature_c', 0, 10),
            'random_strength': trial.suggest_float('random_strength_c', 0, 10),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_c', 1, 100),
        }

        params_t = {
            'depth': trial.suggest_int('depth_t', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate_t', 0.01, 0.3, log=True),
            'iterations': trial.suggest_int('iterations_t', 300, 1500),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg_t', 1, 20),
            'bagging_temperature': trial.suggest_float('bagging_temperature_t', 0, 10),
            'random_strength': trial.suggest_float('random_strength_t', 0, 10),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf_t', 1, 100),
        }

        X_c = X_train[treatment_train == 0][selected_features]
        y_c = y_train[treatment_train == 0]['response_att']

        X_t = X_train[treatment_train == 1][selected_features]
        y_t = y_train[treatment_train == 1]['response_att']

        model_c = base_model.__class__(**{**params_c, **base_params})
        model_c.fit(X_c, y_c)

        model_t = base_model.__class__(**{**params_t, **base_params})
        model_t.fit(X_t, y_t)

        _, auuc_val = get_uplift(
            model_t,
            model_c,
            X_val,
            treatment_val,
            y_val,
            selected_features
        )

        return auuc_val.item()


    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params

    params_c = {
        'depth': best_params['depth_c'],
        'learning_rate': best_params['learning_rate_c'],
        'iterations': best_params['iterations_c'],
        'l2_leaf_reg': best_params['l2_leaf_reg_c'],
        'bagging_temperature': best_params['bagging_temperature_c'],
        'random_strength': best_params['random_strength_c'],
        'min_data_in_leaf': best_params['min_data_in_leaf_c'],
    }

    params_t = {
        'depth': best_params['depth_t'],
        'learning_rate': best_params['learning_rate_t'],
        'iterations': best_params['iterations_t'],
        'l2_leaf_reg': best_params['l2_leaf_reg_t'],
        'bagging_temperature': best_params['bagging_temperature_t'],
        'random_strength': best_params['random_strength_t'],
        'min_data_in_leaf': best_params['min_data_in_leaf_t'],
    }

    X_c = X_train[treatment_train == 0][selected_features]
    y_c = y_train[treatment_train == 0]['response_att']

    X_t = X_train[treatment_train == 1][selected_features]
    y_t = y_train[treatment_train == 1]['response_att']

    model_c = base_model.__class__(**{**params_c, **base_params})
    model_c.fit(X_c, y_c)

    model_t = base_model.__class__(**{**params_t, **base_params})
    model_t.fit(X_t, y_t)

    uplift_pred_train, auuc_train = get_uplift(
        model_t,
        model_c,
        X_train,
        treatment_train,
        y_train['response_att'],
        selected_features
    )

    return model_t, model_c, uplift_pred_train, auuc_train, study

### Customer feature selection from high‐dimensional bank direct marketing data for uplift modeling.

Авторы данной статьи предлагают двухстадийный подход к отбору фичей:

1. Первый этап - выделение релевантных признаков с помощью Bin-based Divergence Filters. (Из статьи Feature Selection Methods for Uplift Modeling and Heterogeneous Treatment Effect). К уже сущесствующим методам авторы предлагают еще два -
    1. UC filter - данные разбиваются на K Бинов на основе перцентилей значений признака. Для каждого Бина вычисляются - сумма значений целевой переменной в treatment группе для данного бина (фактически, количество положительных исходов в treatment); сумма значений целевой переменной в control группе для данного бина; количество наблюдений в treatment группе в данном бине;  количество наблюдений в control группе в данном бине. Итоговый score для признака F вычисляется как сумма uplift'ов по всем бинам, взвешенная на долю наблюдений в каждом бине. Высокое значение RSUC говорит о том, что признак хорошо разделяет данные на группы с разным uplift'ом. Такой признак потенциально важен для моделирования гетерогенности эффекта воздействия (CATE).
    2. AUUC filter использует параметрическую uplift-кривую для оценки важности признаков. Все наблюдения сортируются по возрастанию значения признака F. Для каждого t (количество первых наблюдений в отсортированном списке) строится точка на кривой. Функция кривой задается следующим образом: f(F_j, t) = (Y_t^T / N_t^T - Y_t^C / N_t^C) * (N_t^T + N_t^C), где Y_t^T - сумма исходов в treatment группе среди первых t наблюдений, N_t^T - количество наблюдений в treatment группе среди первых t наблюдений, Y_t^C - 	сумма исходов в control группе среди первых t наблюдений, N_t^C - количество наблюдений в control группе среди первых t наблюдений. Итоговый score для признака F - это площадь под построенной кривой. Высокое значение метрики говорит о том, что при сортировке по данному признаку, объекты с высокими значениями признака имеют систематически другой uplift, чем объекты с низкими значениями. Такой признак потенциально является хорошим модификатором эффекта воздействия.
2. Второй этап - анализ избыточности на отобранном наборе признаков. Начиная с первого по значимости признака в релевантном наборе, мы удаляем признаки с более низким рангом, коэффициент корреляции которых с ним превышает заданный параметр 𝜌₀. В данной работе параметр 𝜌₀ установлен равным 0.8.

Из библиотеки upliftml возьмем метод UpliftCurveFilter, он соотвествует второму предложенному в статье методу оценки признаков с помощью uplift-кривой:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import auc

def rsaauuc_single_feature(
    x: pd.Series,
    treatment: pd.Series,
    y: pd.Series,
    n_bins=10
):
    df = pd.DataFrame({
        'x': x,
        'treatment': treatment,
        'y': y
    }).dropna()

    if df['x'].nunique() < 3:
        return 0.0

    try:
        df = df.sort_values('x')
        df['bin'] = pd.qcut(df['x'], q=n_bins, duplicates='drop')
    except ValueError:
        return 0.0

    uplift_values = []

    for _, group in df.groupby('bin', observed=True):
        yt = group.loc[group.treatment == 1, 'y']
        yc = group.loc[group.treatment == 0, 'y']

        if len(yt) < 5 or len(yc) < 5:
            continue

        uplift_values.append(yt.mean() - yc.mean())

    if len(uplift_values) < 2:
        return 0.0

    return auc(np.arange(len(uplift_values)), uplift_values)

In [ ]:
from tqdm import tqdm


def rsaauuc_feature_ranking(
    X: pd.DataFrame,
    treatment: pd.Series,
    y: pd.Series,
    n_bins=10
):
    scores = {}

    for col in tqdm(X.columns):
        scores[col] = rsaauuc_single_feature(
            X[col], treatment, y, n_bins=n_bins
        )

    return (
        pd.Series(scores)
        .sort_values(ascending=False)
    )

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


def redundancy_reduction(
    X: pd.DataFrame,
    scores: pd.Series,
    top_k=80,
    corr_threshold=0.85
):
    top_feats = scores.head(top_k).index.tolist()

    corr = X[top_feats].corr().abs()

    dist = 1 - corr
    condensed = squareform(dist.values)

    Z = linkage(condensed, method='average')

    clusters = fcluster(Z, t=1 - corr_threshold, criterion='distance')

    cluster_df = pd.DataFrame({
        'feature': top_feats,
        'cluster': clusters,
        'score': scores.loc[top_feats].values
    })

    selected = (
        cluster_df
        .sort_values('score', ascending=False)
        .groupby('cluster')
        .head(1)['feature']
        .tolist()
    )

    return selected, cluster_df

In [ ]:
def get_features(
    X, treatment, y,
    n_bins=10,
    top_k=80,
    corr_threshold=0.80
):
    scores = rsaauuc_feature_ranking(
        X, treatment, y, n_bins=n_bins
    )

    selected, cluster_df = redundancy_reduction(
        X, scores,
        top_k=top_k,
        corr_threshold=corr_threshold
    )

    return selected, scores, cluster_df

In [ ]:
 selected_features_v1, scores_v1, cluster_df_v1 = get_features(X_train, treatment_train, y_train)

In [ ]:
len(selected_features_v1)

In [ ]:
model_t_v1, model_c_v1, uplift_pred_train_v1, auuc_train_v1, study_v1 = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train,
                                                                             X_val, treatment_val, y_val,
                                                                             selected_features_v1)

In [ ]:
uplift_pred_test_v1, auuc_test_v1 = get_uplift(model_t_v1, model_c_v1, X_test, treatment_test, y_test, selected_features_v1)

In [ ]:
# 0.5764277840921775 - до оптуны

### Doubly Robust Adaptive LASSO for Effect

In [ ]:
from catboost import CatBoostRegressor, CatBoostClassifier


def train_nuisance_models(X, T, y):
    mu1 = CatBoostRegressor(verbose=False, random_state=SEED)
    mu0 = CatBoostRegressor(verbose=False, random_state=SEED)
    e = CatBoostClassifier(verbose=False, random_state=SEED)

    mu1.fit(X[T==1], y[T==1])
    mu0.fit(X[T==0], y[T==0])
    e.fit(X, T)

    return mu1, mu0, e

In [ ]:
def compute_dr_pseudo_outcome(X, T, y, mu1, mu0, e):
    mu1_pred = mu1.predict(X)
    mu0_pred = mu0.predict(X)
    e_pred = np.clip(e.predict_proba(X)[:,1], 1e-3, 1-1e-3)

    tau_hat = (
        T * (y - mu1_pred) / e_pred
        - (1-T) * (y - mu0_pred) / (1 - e_pred)
        + mu1_pred - mu0_pred
    )

    return tau_hat

In [ ]:
from sklearn.linear_model import Ridge

def initial_ridge_weights(X, tau_hat):
    ridge = Ridge(alpha=1.0, random_state=SEED)
    ridge.fit(X, tau_hat)
    return np.abs(ridge.coef_)

In [ ]:
from sklearn.linear_model import Lasso

def adaptive_lasso(X, tau_hat, weights, alpha=0.01):
    X_scaled = X / (weights + 1e-6)

    lasso = Lasso(alpha=alpha, random_state=SEED)
    lasso.fit(X_scaled, tau_hat)

    beta = lasso.coef_ / (weights + 1e-6)

    return beta


In [ ]:
def dr_adaptive_lasso_feature_selection(
    X, T, y,
    alpha=0.01
):
    mu1, mu0, e = train_nuisance_models(X, T, y)

    tau_hat = compute_dr_pseudo_outcome(X, T, y, mu1, mu0, e)

    weights = initial_ridge_weights(X, tau_hat)

    beta = adaptive_lasso(X, tau_hat, weights, alpha)

    selected_features = X.columns[np.abs(beta) > 1e-6].tolist()

    return selected_features, beta, tau_hat


In [ ]:
selected_features_v2, beta, tau_hat = dr_adaptive_lasso_feature_selection(X_train, treatment_train, y_train, alpha=0.001)

In [ ]:
len(selected_features_v2)

In [ ]:
model_t_v2, model_c_v2, uplift_pred_train_v2, auuc_train_v2, study_v2 = train_tlearner(BASE_MODEL, 
                                                                                       X_train, 
                                                                                       treatment_train, 
                                                                                       y_train, 
                                                                                       X_val, 
                                                                                       treatment_val, 
                                                                                       y_val, 
                                                                                       selected_features_v2)

In [ ]:
uplift_pred_test_v2, auuc_test_v2 = get_uplift(model_t_v2, model_c_v2, X_test, treatment_test, y_test, 
                                               selected_features_v2)

In [ ]:
# до оптуны метрика была выше - 0.6668880309490919

### A Non-parametric Bayesian Approach for Uplift Discretization and Feature Selection

Код реализован в библиотеке kuplift - https://github.com/UData-Orange/kuplift/blob/main/kuplift/feature_selection.py

Для ускорения работы отберем признаки на валидационной выборке

In [ ]:
import kuplift as kp

In [ ]:
fs = kp.FeatureSelection()

important_vars = fs.filter(
    X_val,
    treatment_val,
    y_val
)

In [ ]:
important_vars

In [ ]:
len(important_vars)

In [ ]:
scores = sorted(important_vars.values(), reverse=True)

plt.plot(scores)
plt.title('Feature importance')
plt.show()

In [ ]:
def find_elbow(scores_dict):
    scores = np.array(sorted(scores_dict.values(), reverse=True))

    x = np.arange(len(scores))
    y = scores

    line_start = np.array([0, y[0]])
    line_end = np.array([len(scores)-1, y[-1]])

    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.linalg.norm(line_vec)

    distances = []
    for i in range(len(scores)):
        point = np.array([i, y[i]])
        vec = point - line_start
        proj = np.dot(vec, line_vec_norm) * line_vec_norm
        dist = np.linalg.norm(vec - proj)
        distances.append(dist)

    elbow_idx = np.argmax(distances)
    return elbow_idx + 1

optimal_features = find_elbow(important_vars)
print(f'Оптимальное количество признаков по локтю: {optimal_features}')

In [ ]:
scores_series = pd.Series(important_vars).sort_values(ascending=False)
top_features = scores_series.head(optimal_features).index.tolist()

In [ ]:
top_features

In [ ]:
model_t_v3, model_c_v3, uplift_pred_train_v3, auuc_train_v3, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                             X_val, treatment_val, y_val, 
                                                                             top_features)

In [ ]:
uplift_pred_test_v3, auuc_test_v3 = get_uplift(model_t_v3, model_c_v3, X_test, treatment_test, y_test, top_features)

In [ ]:
# попробуем взять топ-50 фичей
top_features_v2 = scores_series.head(50).index.tolist()
model_t_v4, model_c_v4, uplift_pred_train_v4, auuc_train_v4, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                             X_val, treatment_val, y_val, 
                                                                             top_features_v2)

In [ ]:
uplift_pred_test_v4, auuc_test_v4 = get_uplift(model_t_v4, model_c_v4, X_test, treatment_test, y_test, top_features_v2)

### Robustness-enhanced Uplift Modeling with Adversarial Feature Desensitization

In [ ]:
class RUAD_AUUC_Selector:
    def __init__(self, hidden_dim=128, epochs=50, lr=1e-3, top_k=30, device='cpu'):
        
        self.hidden_dim = hidden_dim
        self.epochs = epochs
        self.lr = lr
        self.top_k = top_k
        self.device = device

    def _create_joint_label(self, t, y):
        
        return t * 2 + y

    def _compute_uplift(self, probs):
  
        p_y1_t0 = probs[:,1] / (probs[:,0]+probs[:,1]+1e-8)
        p_y1_t1 = probs[:,3] / (probs[:,2]+probs[:,3]+1e-8)
        tau_hat = p_y1_t1 - p_y1_t0
        return tau_hat

    def _auuc_per_feature(self, X, uplift):
        
        N, d = X.shape
        scores = []
        for i in range(d):
            order = np.argsort(X[:,i])
            uplift_sorted = uplift[order]
            if len(uplift_sorted) < 2:
                scores.append(0)
            else:
                scores.append(auc(np.arange(len(uplift_sorted)), uplift_sorted))
        return np.array(scores)

    def fit(self, X_train, t_train, y_train, X_val, t_val, y_val):
        
        if isinstance(X_train, pd.DataFrame):
            self.feature_names = X_train.columns
            X_train_np = X_train.values
            X_val_np = X_val.values
        else:
            self.feature_names = np.arange(X_train.shape[1])
            X_train_np = X_train
            X_val_np = X_val

        d = X_train_np.shape[1]

        X_tensor = torch.tensor(X_train_np, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(self._create_joint_label(t_train, y_train), dtype=torch.long).to(self.device)

        self.model = nn.Sequential(
            nn.Linear(d, self.hidden_dim),
            nn.ReLU(),
            nn.Linear(self.hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        ).to(self.device)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.CrossEntropyLoss()

        losses = []
        auuc_history = []

        for epoch in range(self.epochs):
    
            logits = self.model(X_tensor)
            loss = criterion(logits, y_tensor)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                probs_val = torch.softmax(self.model(torch.tensor(X_val_np, dtype=torch.float32).to(self.device)), dim=1).cpu().numpy()
            
            uplift_val = self._compute_uplift(probs_val)
            auuc_epoch = self._auuc_per_feature(X_val_np, uplift_val)
            auuc_history.append(auuc_epoch)

            clear_output(wait=True)
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(losses)+1), losses, marker='o')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title('RUAD Training Loss')
            plt.grid(True)
            plt.show()

        self.importance_ = np.mean(auuc_history, axis=0)
        idx = np.argsort(self.importance_)[-self.top_k:]
        self.selected_idx_ = idx
        self.selected_features_ = self.feature_names[idx]

        return self

    def transform(self, X):
        
        if isinstance(X, pd.DataFrame):
            return X[self.selected_features_]
        return X[:, self.selected_idx_]
    
    def fit_transform(self, X_train, t_train, y_train, X_val, t_val, y_val):
        
        self.fit(X_train, t_train, y_train, X_val, t_val, y_val)
        return self.transform(X_train)

In [ ]:
selector = RUAD_AUUC_Selector(hidden_dim=128, epochs=80, lr=1e-3, top_k=50)
X_selected = selector.fit_transform(X_train, treatment_train, y_train, X_val, treatment_val, y_val)

In [ ]:
print('Число отобранных признако:', len(selector.selected_features_))

In [ ]:
selected_features_v5 = X_selected.columns.tolist()

In [ ]:
model_t_v5, model_c_v5, uplift_pred_train_v5, auuc_train_v5, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                             X_val, treatment_val, y_val, 
                                                                             selected_features_v5)

In [ ]:
uplift_pred_test_v5, auuc_test_v5 = get_uplift(model_t_v5, model_c_v5, X_test, treatment_test, y_test, selected_features_v5)

### Multihead Causal Distilling Weighting Is All You Need for Uplift Modeling

In [ ]:
class CausalFeatureSelector(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.mask = nn.Parameter(torch.ones(input_dim))

    def forward(self, x):

        mask = torch.sigmoid(self.mask)

        return x * mask

class MCDWModel(nn.Module):

    def __init__(self, input_dim, hidden=128, heads=5):

        super().__init__()

        self.selector = CausalFeatureSelector(input_dim)

        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )

        self.heads = nn.ModuleList([
            nn.Linear(hidden, 1) for _ in range(heads)
        ])

    def forward(self, x):

        x = self.selector(x)

        h = self.backbone(x)

        preds = torch.cat([head(h) for head in self.heads], dim=1)

        return preds

class MCDWFeatureSelector:

    def __init__(
        self,
        hidden=128,
        heads=5,
        epochs=50,
        lr=1e-3,
        lambda_l1=1e-3,
        device='cpu'
    ):

        self.hidden = hidden
        self.heads = heads
        self.epochs = epochs
        self.lr = lr
        self.lambda_l1 = lambda_l1
        self.device = device


    def fit(self, X, t, y):

        if isinstance(X, pd.DataFrame):

            self.feature_names = X.columns
            X = X.values

        else:

            self.feature_names = np.arange(X.shape[1])

        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).to(self.device)
        t_tensor = torch.tensor(t, dtype=torch.float32).to(self.device)

        input_dim = X.shape[1]

        self.model = MCDWModel(input_dim, self.hidden, self.heads).to(self.device)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        losses = []

        for epoch in range(self.epochs):

            preds = self.model(X_tensor)

            target = y_tensor * (2 * t_tensor - 1)

            mse = ((preds - target.unsqueeze(1)) ** 2).mean()

            mask = torch.sigmoid(self.model.selector.mask)

            l1 = mask.abs().mean()

            loss = mse + self.lambda_l1 * l1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

            clear_output(wait=True)
            plt.figure(figsize=(6,4))
            plt.plot(range(1, len(losses)+1), losses, marker='o')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title('MCDW Training Loss')
            plt.grid(True)
            plt.show()

        mask_values = torch.sigmoid(self.model.selector.mask).detach().cpu().numpy()

        self.importance_ = mask_values

        return self


    def get_feature_importance(self):
        
        return pd.Series(self.importance_, index=self.feature_names).sort_values(ascending=False)


    def select_top_k(self, k):

        imp = self.get_feature_importance()
        return imp.head(k).index.tolist()


    def transform(self, X, k):

        features = self.select_top_k(k)

        if isinstance(X, pd.DataFrame):
            return X[features]

        else:
            idx = [np.where(self.feature_names == f)[0][0] for f in features]

            return X[:, idx]

In [ ]:
selector = MCDWFeatureSelector(
    hidden=256,
    heads=5,
    epochs=30,
    lr=1e-3
)


In [ ]:
selector.fit(X_train, treatment_train, y_train)

In [ ]:
importance = selector.get_feature_importance()

In [ ]:
top_50_features = selector.select_top_k(50)

In [ ]:
top_30_features = selector.select_top_k(30)

In [ ]:
top_80_features = selector.select_top_k(80)

In [ ]:
top_50_features

In [ ]:
model_t_v6, model_c_v6, uplift_pred_train_v6, auuc_train_v6, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                                    X_val, treatment_val, y_val,
                                                                                    top_30_features)

In [ ]:
uplift_pred_test_v6, auuc_test_v6 = get_uplift(model_t_v6, model_c_v6, X_test, treatment_test, y_test, top_30_features)

In [ ]:
model_t_v7, model_c_v7, uplift_pred_train_v7, auuc_train_v7, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                                    X_val, treatment_val, y_val,
                                                                                    top_50_features)

In [ ]:
uplift_pred_test_v7, auuc_test_v7 = get_uplift(model_t_v7, model_c_v7, X_test, treatment_test, y_test, top_50_features)

In [ ]:
# 128 hidden_dim - 0.5173856389513593 test

In [ ]:
model_t_v8, model_c_v8, uplift_pred_train_v8, auuc_train_v8, study = train_tlearner(BASE_MODEL, X_train, treatment_train, y_train, 
                                                                                    X_val, treatment_val, y_val,
                                                                                    top_80_features)

In [ ]:
uplift_pred_test_v8, auuc_test_v8 = get_uplift(model_t_v8, model_c_v8, X_test, treatment_test, y_test, top_80_features)

### Вывод

|Статья |Число признаков| AUUC train |AUUC test |
|-------|-----------|----------|-------|
Customer feature selection from high‐dimensional bank direct marketing data for uplift modeling| 48 |  6.72| 0.74 |
Doubly Robust Adaptive LASSO for Effect| 184 |  6.74| 0.49|
A Non-parametric Bayesian Approach for Uplift Discretization and Feature Selection| 12 |  7.03| 0.81|
A Non-parametric Bayesian Approach for Uplift Discretization and Feature Selection| 50 |  5.97| 0.46|
Robustness-enhanced Uplift Modeling with Adversarial Feature Desensitization| 50 |  2.6| 0.50|
Multihead Causal Distilling Weighting Is All You Need for Uplift Modeling| 30 | 6.70| 0.57|
Multihead Causal Distilling Weighting Is All You Need for Uplift Modeling| 50 | 6.77| 0.41|
Multihead Causal Distilling Weighting Is All You Need for Uplift Modeling| 80 | 10.49| 0.50|

Наилучший результат показал эксперимент из статьи A Non-parametric Bayesian Approach for Uplift Discretization and Feature Selection с 12 признаками    